# Grandes modelos de lenguaje (LLMs)

En este notebook, exploraremos el uso de grandes modelos de lenguaje (LLMs) para tareas de procesamiento de lenguaje natural, con hincapié en aplicaciones clínicas y de salud. Cubriremos cómo conectarse a APIs de LLMs, enviar consultas, procesar respuestas y algunos ejemplos de prompting y buenas prácticas.

In [1]:
import requests
from openai import OpenAI
from pydantic import BaseModel
import json
import datasets
from sklearn.metrics import classification_report

/home/vscode/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Autenticación

Para interactuar con un LLM, primero debemos autenticar nuestra aplicación. Esto generalmente implica obtener una clave API de un proveedor de LLM. Asegúrate de tener tu clave API a mano.

In [ ]:
API_KEY = ""

## API de OpenAI

A través de consultas HTTP a la API de OpenAI, podemos enviar solicitudes y recibir respuestas de un modelo de lenguaje. Aquí hay un ejemplo básico de cómo hacerlo en Python.

In [9]:
url = "https://models.villena.cl/v1/responses"
headers = {
    "Content-Type": "application/json",
}
headers["Authorization"] = f"Bearer {API_KEY}"
data = {
    "model": "gpt-4.1-nano",
    "input": "Define qué es el procesamiento de lenguaje natural clínico.",
}
response = requests.post(url, headers=headers, json=data)
response.json()

{'id': 'resp_bGl0ZWxsbTpjdXN0b21fbGxtX3Byb3ZpZGVyOm9wZW5haTttb2RlbF9pZDo5ZjU1MDFkMC1lM2Y0LTQzNzEtOWNlZS0xM2JhY2ZlNDc5MzI7cmVzcG9uc2VfaWQ6cmVzcF8wZTA0MTgyYTc4M2Y2N2I0MDA2OTMxOWRhNmRjYmM4MTk1ODAxMzk1NGQxMWJjYjI1NA==',
 'created_at': 1764859302,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [{'id': 'msg_0e04182a783f67b40069319da856a08195b443f5116d7845ad',
   'content': [{'annotations': [],
     'text': 'El procesamiento de lenguaje natural clínico (PLN clínico) es una rama especializada del procesamiento de lenguaje natural (PLN) que se enfoca en la extracción, interpretación y análisis de información textual generada en contextos clínicos y de atención médica. Este campo busca convertir el lenguaje humano no estructurado, como notas de progreso, informes médicos, registros electrónicos de salud y otra documentación clínica, en datos estructurados y utilizables que puedan apoyar l

## Uso de la biblioteca `openai`

En vez de hacer solicitudes HTTP manualmente, podemos usar la biblioteca `openai` para simplificar el proceso. Asegúrate de instalarla primero:


In [10]:
client = OpenAI(
    api_key=API_KEY,
    base_url="https://models.villena.cl",
)

###  Uso básico

Podemos usar la biblioteca `openai` para enviar solicitudes a la API de OpenAI. Aquí hay un ejemplo básico de cómo hacerlo

In [11]:
response = client.responses.create(
    model="gpt-4o",
    instructions="Eres un experto en procesamiento de lenguaje natural clínico. Responde a la pregunta de manera clara y concisa.",
    input="¿Qué debo tomar en cuenta para desarrollar una función de preprocesamiento?",
)

print(response.output_text)

Al desarrollar una función de preprocesamiento para datos clínicos en procesamiento de lenguaje natural (PLN), considera los siguientes aspectos:

1. **Cumplimiento de normativas**: Asegúrate de que el manejo de los datos cumple con las regulaciones de privacidad y seguridad, como HIPAA o GDPR.

2. **Anonimización**: Elimina o codifica información identificable para proteger la privacidad del paciente.

3. **Normalización del texto**: Convierte el texto a minúsculas, elimina caracteres especiales y corrige errores tipográficos.

4. **Segmentación de oraciones y palabras**: Utiliza técnicas adecuadas para dividir el texto en oraciones y palabras correctamente, dado el uso frecuente de abreviaturas y términos especializados.

5. **Eliminación de stopwords**: Retira palabras comunes que no aportan significado (aunque en textos médicos, algunas pueden ser relevantes, así que evalúa bien esta acción).

6. **Lematización y stemming**: Reduce las palabras a su raíz base para un análisis más e

También podemos usar imágenes como parte de nuestras consultas. A continuación, se muestra un ejemplo de cómo enviar una imagen junto con un mensaje de texto.


In [12]:
prompt = "¿Qué enfermedad es probable que tenga el paciente?"
img_url = "https://patoral.umayor.cl/canmucor/ca_leng_mb1.jpg"

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"{img_url}"},
            ],
        }
    ],
)

print(response.output_text)

Lo siento, pero no puedo proporcionar un diagnóstico médico basado en una imagen. Sin embargo, si hay síntomas específicos o antecedentes médicos que deseas discutir, puedo intentar ayudarte con información general. Es importante que un profesional de la salud evalúe cualquier situación médica.


### Mensajes y roles

Los mensajes en la API de OpenAI tienen roles que indican quién está hablando. Los roles comunes son `system`, `user` y `assistant`. Aquí hay un ejemplo de cómo estructurar un mensaje

In [13]:
response = client.responses.create(
    model="gpt-4.1",
    input=[
        {"role": "developer", "content": "Eres un experto en salud digital."},
        {"role": "user", "content": "Qué es lo más importante al implementar un proyecto de informática médica?"}
    ]
)
print(response.output_text)


La implementación de un proyecto de informática médica es un proceso complejo que requiere equilibrar aspectos tecnológicos, clínicos, organizacionales y humanos. Lo **más importante** para el éxito es **alinear la tecnología con las necesidades clínicas y organizacionales**, asegurando que el proyecto aporte valor real y mejore los procesos de atención.

A continuación se resumen los pilares fundamentales:

1. **Enfoque centrado en el usuario (clínico y paciente)**  
   – Involucrar desde el inicio a los usuarios finales (médicos, enfermeros, administrativos, pacientes).  
   – Recoger y priorizar sus necesidades para diseñar flujos de trabajo útiles y adoptables.

2. **Interoperabilidad y estándares**  
   – Asegurar que la solución pueda integrarse con otros sistemas existentes (HIS, LIS, PACS, etc.).  
   – Usar estándares reconocidos (HL7, FHIR, DICOM, SNOMED CT, LOINC).

3. **Gestión del cambio y capacitación**  
   – Implementar estrategias claras para gestionar la resistencia a

In [14]:
developer_message = """
# Identidad

Eres un priorizador de la lista de espera chilena y deben priorizar pacientes en las categorías Urgente o Rutina.

# Instrucciones

* Solo responde con una palabra: "Urgente" o "Rutina".
"""

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {"role": "developer", "content": developer_message},
        {"role": "user", "content": "El paciente presenta un dolor muy leve en el dedo meñique izquierdo."}
    ]
)
print(response.output_text)


Rutina


### Adición de ejemplos

Para mejorar la calidad de las respuestas, podemos proporcionar ejemplos de preguntas y respuestas esperadas. Esto ayuda al modelo a entender mejor el contexto y las expectativas.

In [15]:
developer_message = """
# Identidad

Eres un priorizador de la lista de espera chilena y deben priorizar pacientes en las categorías Urgente o Rutina.

# Instrucciones

* Solo responde con una palabra: "Urgente" o "Rutina".

# Ejemplos

<interconsulta>
El paciente presenta pérdida de peso de 10 kg en los últimos 3 meses y tiene antecedentes de cáncer de colon.
</interconsulta>
<assistant_response">
Urgente
</assistant_response>
<interconsulta>
El paciente presenta un dolor muy leve en el dedo meñique izquierdo.
</interconsulta>
<assistant_response">
Rutina
</assistant_response>
<interconsulta>
El paciente tiene un dolor intenso en el pecho y dificultad para respirar.
</interconsulta>
<assistant_response">
Urgente
</assistant_response>
"""

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {"role": "developer", "content": developer_message},
        {"role": "user", "content": "El paciente tiene un dolor intenso en la zona parietal izquierda."}
    ]
)
print(response.output[0].content[0].text)


Urgente


### Respuesta estructurada

Para obtener respuestas más estructuradas, podemos usar el parámetro `text_format` en la solicitud. Esto nos permite especificar el formato de la respuesta esperada.

In [16]:
# Definir el esquema de salida estructurada
class InterconsultaPrioridad(BaseModel):
    prioridad: str  # "Urgente" o "Rutina"

# Usar el método parse para obtener una respuesta estructurada
response = client.responses.parse(
    model="gpt-4o-2024-08-06",
    input=[
        {"role": "system", "content": "Eres un priorizador de la lista de espera chilena. Responde solo con la prioridad del paciente ('Urgente' o 'Rutina')."},
        {"role": "user", "content": "El paciente tiene fiebre alta y dificultad respiratoria."}
    ],
    text_format=InterconsultaPrioridad,
)

print(response.output_parsed.prioridad)

Urgente


En algunos casos, es fundamental que la respuesta del modelo siga una estructura específica y validable, especialmente para integraciones clínicas o flujos automatizados. La API de OpenAI permite definir un esquema JSON (JSON Schema) que el modelo debe seguir estrictamente en su respuesta. A continuación, se muestra cómo definir un esquema para priorización de interconsultas y cómo solicitar al modelo que responda usando exactamente ese formato estructurado.

In [17]:
schema = {
    "type": "object",
    "properties": {
        "prioridad": {
            "type": "string",
            "enum": ["Urgente", "Rutina"],
            "description": "Prioridad del paciente según la interconsulta"
        },
        "descripcion": {
            "type": "string",
            "description": "Descripción adicional de la interconsulta"
        }
    },
    "required": ["prioridad", "descripcion"],
    "additionalProperties": False
}

response = client.responses.create(
    model="gpt-4o",
    input=[
        {"role": "system", "content": "Eres un priorizador de la lista de espera chilena. Responde solo con la prioridad del paciente ('Urgente' o 'Rutina')."},
        {"role": "user", "content": "El paciente refiere dolor abdominal leve desde hace 2 semanas."}
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "prioridad_interconsulta",
            "schema": schema,
            "strict": True
        }
    }
)

json.loads(response.output[0].content[0].text)

{'prioridad': 'Rutina',
 'descripcion': 'Dolor abdominal leve sin otros síntomas de alarma.'}

### Chain of Thought (Cadena de Pensamiento)

El prompting tipo "chain of thought" (cadena de pensamiento) le indica al modelo que razone paso a paso antes de dar una respuesta final. Esto es útil para tareas complejas donde se requiere justificar o explicar el razonamiento detrás de la decisión.

A continuación, se muestra un ejemplo donde se le pide al modelo que explique su razonamiento antes de priorizar la interconsulta


In [18]:
cot_prompt = """
Eres un priorizador de la lista de espera chilena. 
Primero, analiza los síntomas del paciente paso a paso y explica tu razonamiento. 
Luego, responde con la prioridad final: "Urgente" o "Rutina".

Paciente: El paciente presenta fiebre alta, tos persistente y dificultad respiratoria.
"""

response = client.responses.create(
    model="gpt-4o",
    input=[
        {"role": "system", "content": cot_prompt}
    ]
)

print(response.output_text)

Veamos cada uno de los síntomas:

1. **Fiebre alta:** La fiebre es una respuesta común del cuerpo ante infecciones. Una fiebre alta puede ser indicativa de una infección más severa o un proceso inflamatorio significativo.

2. **Tos persistente:** La tos es un mecanismo del cuerpo para despejar las vías respiratorias. Una tos persistente puede indicar una infección respiratoria, como neumonía o bronquitis.

3. **Dificultad respiratoria:** Este es un síntoma más preocupante, puesto que puede señalar una obstrucción de las vías respiratorias o una insuficiencia para intercambiar gases adecuadamente en los pulmones. Es un signo que, en combinación con fiebre alta y tos, sugiere una posible infección respiratoria severa.

**Razón de priorización:**

- La combinación de fiebre alta, tos persistente y dificultad respiratoria podría indicar una infección respiratoria grave o una condición que afecta la función pulmonar de manera significativa.
- La dificultad respiratoria requiere atención inm

## Clasificador utilizando LLMs

Los LLMs también pueden ser utilizados como clasificadores para tareas específicas, como la clasificación de interconsultas médicas. A continuación, se muestra un ejemplo de cómo implementar un clasificador utilizando un LLM.

In [19]:
spanish_diagnostics = datasets.load_dataset('fvillena/spanish_diagnostics')

Generating test split: 100%|██████████| 30000/30000 [00:00<00:00, 2033207.62 examples/s]


In [20]:
test = spanish_diagnostics['test'].select(range(100))

In [21]:
def classifier(text):
    schema = {
        "type": "object",
        "properties": {
            "tipo": {
                "type": "string",
                "enum": ["dental", "no_dental"],
                "description": "Tipo de diagnóstico: 'dental' cuando se debe enviar la interconsulta a una especialidad dental o 'no_dental' cuando no se debe enviar a una especialidad dental"
            }
        },
        "required": ["tipo"],
        "additionalProperties": False
    }
    response = client.responses.create(
        model="gpt-4o",
        input=[
            {"role": "system", "content": "Eres un clasificador de diagnósticos médicos."},
            {"role": "user", "content": text}
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "diagnostico_clasificacion",
                "schema": schema,
                "strict": True
            }
        }
    )
    return json.loads(response.output[0].content[0].text)["tipo"]

In [22]:
classifier("caries en el diente 12 y 13, dolor leve al masticar")

'dental'

In [23]:
predicted = [classifier(item['text']) for item in test]

In [24]:
print(classification_report(["dental" if item['label'] == 1 else "no_dental" for item in test], predicted))

              precision    recall  f1-score   support

      dental       0.93      0.98      0.96        44
   no_dental       0.98      0.95      0.96        56

    accuracy                           0.96       100
   macro avg       0.96      0.96      0.96       100
weighted avg       0.96      0.96      0.96       100



In [25]:
def classifier_few_shot(text):
    schema = {
        "type": "object",
        "properties": {
            "tipo": {
                "type": "string",
                "enum": ["dental", "no_dental"],
                "description": "Tipo de diagnóstico: 'dental' cuando se debe enviar la interconsulta a una especialidad dental o 'no_dental' cuando no se debe enviar a una especialidad dental"
            }
        },
        "required": ["tipo"],
        "additionalProperties": False
    }
    response = client.responses.create(
        model="gpt-4o",
        input=[
            {"role": "system", "content": "Eres un clasificador de diagnósticos médicos."},
            {"role": "user", "content": "paciente con dolor en el diente 12 y 13"},
            {"role": "assistant", "content": '{"tipo":"dental"}'},
            {"role": "user", "content": "paciente con dolor en la rodilla derecha"},
            {"role": "assistant", "content": '{"tipo":"no_dental"}'},
            {"role": "user", "content": text}
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "diagnostico_clasificacion",
                "schema": schema,
                "strict": True
            }
        }
    )
    return json.loads(response.output[0].content[0].text)["tipo"]

In [26]:
classifier_few_shot("El paciente presenta dolor en el diente 12 y 13, con sensibilidad al frío y al calor.")

'dental'

In [27]:
predicted = [classifier_few_shot(item['text']) for item in test]

In [28]:
print(classification_report(["dental" if item['label'] == 1 else "no_dental" for item in test], predicted))

              precision    recall  f1-score   support

      dental       0.92      1.00      0.96        44
   no_dental       1.00      0.93      0.96        56

    accuracy                           0.96       100
   macro avg       0.96      0.96      0.96       100
weighted avg       0.96      0.96      0.96       100

